# 04 - Action：分析数据

> **STAR法则的第四步**：清洗数据，执行统计分析，逐步完善结论。

---

## 📋 学习目标

完成本notebook后，你将能够：
1. 清洗脏数据
2. 执行描述性统计
3. 进行假设检验
4. 计算效应量和置信区间
5. 处理多重检验问题

---

## 🧹 数据清洗

### 加载数据

```python
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# 加载原始数据
df = pd.read_csv("../data/raw/experiment_data.csv")

print("数据加载完成")
print(f"形状: {df.shape}")
print(f"\n缺失值统计：")
print(df.isnull().sum())
```

### 清洗逻辑

**TODO 1**：实现数据清洗函数。

```python
def clean_data(df):
    """
    清洗实验数据
    
    策略：
    1. 去重：保留首次出现的用户
    2. 缺失值填充：用历史均值或全局均值
    3. 异常值处理：截断到合理范围
    4. 校验：确保分组只有control/treatment
    """
    df_clean = df.copy()
    
    print("数据清洗过程：")
    print("="*60)
    
    # TODO 1.1：去重
    print("1. 去重...")
    # 提示：使用drop_duplicates，保留第一次出现
    
    # TODO 1.2：处理缺失值
    print("\n2. 处理缺失值...")
    # 提示：
    # - dwell_time缺失：用historical_dwell_time均值填充
    # - ctr缺失：用0填充
    # - retention_next_day缺失：用全局留存率填充
    
    # TODO 1.3：处理异常值
    print("\n3. 处理异常值...")
    # 提示：
    # - dwell_time > 7200秒（2小时）：用99分位数截断
    # - ctr > 0.5：用0.5截断
    
    # TODO 1.4：校验分组
    print("\n4. 校验分组...")
    # 提示：检查group列是否只有control和treatment
    
    print("\n" + "="*60)
    print("清洗完成")
    print(f"最终数据形状: {df_clean.shape}")
    
    return df_clean

df_clean = clean_data(df)

# 保存清洗后的数据
df_clean.to_csv("../data/processed/experiment_data_cleaned.csv", index=False)
print("\n清洗后数据已保存")
```

---

## 📊 描述性统计

```python
print("\n描述性统计 - 按分组：")
print("="*60)

metrics = ["dwell_time", "ctr", "retention_next_day"]

for metric in metrics:
    print(f"\n{metric}:")
    desc = df_clean.groupby("group")[metric].describe()
    print(desc)
    
    # 计算中位数和IQR
    median = df_clean.groupby("group")[metric].median()
    q25 = df_clean.groupby("group")[metric].quantile(0.25)
    q75 = df_clean.groupby("group")[metric].quantile(0.75)
    
    print(f"  中位数: control={median['control']:.2f}, treatment={median['treatment']:.2f}")
    print(f"  IQR: control=[{q25['control']:.2f}, {q75['control']:.2f}], treatment=[{q25['treatment']:.2f}, {q75['treatment']:.2f}]")
```

### 可视化分布

```python
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, metric in enumerate(metrics):
    ax = axes[i]
    
    # 箱线图
    df_clean.boxplot(column=metric, by="group", ax=ax)
    ax.set_title(f"{metric} by group")
    ax.set_xlabel("Group")

plt.suptitle("")  # 移除默认标题
plt.tight_layout()
plt.show()

# 停留时长的分布（右偏）
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 原始分布
axes[0].hist(df_clean["dwell_time"], bins=50, alpha=0.7, edgecolor='black')
axes[0].set_title("Dwell Time Distribution (Original)")
axes[0].set_xlabel("Dwell Time (seconds)")
axes[0].set_ylabel("Frequency")

# 对数变换后的分布
axes[1].hist(np.log(df_clean["dwell_time"]), bins=50, alpha=0.7, edgecolor='black')
axes[1].set_title("Dwell Time Distribution (Log-transformed)")
axes[1].set_xlabel("Log(Dwell Time)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()
```

---

## 🔬 假设检验

### AB检验函数

**TODO 2**：实现完整的AB检验函数。

```python
def ab_test(df, metric, group_col="group", alpha=0.05):
    """
    AB检验：比较两组在指标上的差异
    
    返回:
        dict: 包含检验结果的字典
    """
    control = df[df[group_col] == "control"][metric].dropna()
    treatment = df[df[group_col] == "treatment"][metric].dropna()
    
    # TODO 2.1：执行t检验（假设方差不等）
    # 提示：使用scipy.stats.ttest_ind，equal_var=False
    
    # TODO 2.2：计算效应量（Cohen's d）
    # 提示：d = (mean_treatment - mean_control) / pooled_std
    
    # TODO 2.3：计算置信区间
    # 提示：diff ± z * SE
    
    print(f"AB检验 - {metric}")
    print(f"  对照组: n={len(control)}, mean={control.mean():.2f}, std={control.std():.2f}")
    print(f"  实验组: n={len(treatment)}, mean={treatment.mean():.2f}, std={treatment.std():.2f}")
    # TODO：打印差异、p值、效应量、置信区间
    print(f"  结论: {'显著 ✓' if p_value < alpha else '不显著 ✗'} (α={alpha})")
    print()
    
    return {
        "metric": metric,
        "control_mean": control.mean(),
        "treatment_mean": treatment.mean(),
        "diff": diff,
        "relative_diff": diff/control.mean(),
        "p_value": p_value,
        "cohens_d": cohens_d,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "significant": p_value < alpha
    }

# 执行AB检验
print("="*60)
print("AB检验结果")
print("="*60)

results = []
for metric in metrics:
    result = ab_test(df_clean, metric)
    results.append(result)
```

---

## ⚠️ 多重检验校正

### 问题：如果不校正，假阳性率是多少？

```python
# 假设3个指标独立，每个α=0.05
n_metrics = 3
alpha_single = 0.05

# 至少一个假阳性的概率
false_positive_rate = 1 - (1 - alpha_single) ** n_metrics

print(f"检验指标数: {n_metrics}")
print(f"单个检验α: {alpha_single}")
print(f"至少一个假阳性的概率: {false_positive_rate:.1%}")
print(f"(不是5%，而是{false_positive_rate:.1%}！)")
```

### 校正方法

**TODO 3**：实现多重检验校正。

```python
from statsmodels.stats.multitest import multipletests

# 提取p值
p_values = [r["p_value"] for r in results]
metric_names = [r["metric"] for r in results]

print("原始p值：")
for name, p in zip(metric_names, p_values):
    print(f"  {name}: {p:.4f}")

# TODO 3.1：Bonferroni校正
# 提示：使用multipletests，method='bonferroni'

# TODO 3.2：FDR校正 (Benjamini-Hochberg)
# 提示：使用multipletests，method='fdr_bh'

print("\n" + "="*60)
print("多重检验校正结果")
print("="*60)
print(f"{'指标':<15} {'原始p值':<12} {'Bonferroni':<12} {'FDR':<12}")
print("-"*60)
for i, name in enumerate(metric_names):
    # TODO：打印校正后的p值
    pass

print(f"\nBonferroni校正后显著: {sum(reject_bonf)}个")
print(f"FDR校正后显著: {sum(reject_fdr)}个")
```

### 失败路径：不校正的后果

```python
print("\n" + "="*60)
print("失败路径：忽略多重检验的后果")
print("="*60)

consequence = """
TODO：描述不校正的后果

场景：你同时检验了3个指标，其中1个"显著"（p=0.04）

如果不校正：
1. 你以为这个指标真的显著...
2. 但实际上...
3. 导致...
"""

print(consequence)
```

---

## 📈 结果汇总

```python
# 创建结果汇总表
summary = pd.DataFrame([
    {
        "指标": r["metric"],
        "对照组均值": f"{r['control_mean']:.2f}",
        "实验组均值": f"{r['treatment_mean']:.2f}",
        "绝对差异": f"{r['diff']:.2f}",
        "相对差异": f"{r['relative_diff']*100:.2f}%",
        "p值": f"{r['p_value']:.4f}",
        "Cohen's d": f"{r['cohens_d']:.4f}",
        "95% CI": f"[{r['ci_lower']:.2f}, {r['ci_upper']:.2f}]",
        "显著": "✓" if r["significant"] else "✗"
    }
    for r in results
])

print("\n实验结果汇总：")
print(summary.to_string(index=False))
```

---

## 📝 本节总结

完成本节，你应该已经：

- [ ] 清洗脏数据（缺失值、异常值、重复值）
- [ ] 执行描述性统计和可视化
- [ ] 进行AB假设检验
- [ ] 计算效应量和置信区间
- [ ] 执行多重检验校正
- [ ] 理解不校正的后果

**下一步**：进入 `05-action-iteration.ipynb`，进行迭代调整和延伸分析。

---

> 💡 **关键洞察**：
> p值告诉你"有没有"，效应量告诉你"有多大"。
> 两者缺一不可。
